In [146]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import numpy as np

df_missoes = pd.read_csv('thor_wwii_data_clean.csv', low_memory=False)
df_avioes = pd.read_csv('thor_wwii_aircraft_gloss.csv')
df_armas = pd.read_csv('thor_wwii_weapon_gloss.csv')



#  PIPELINE DE DADOS (INTEGRAÇÃO, LIMPEZA E TRANSFORMAÇÃO)


In [147]:
# Cruzamento das missões com o glossário de aviões, usando a sigla do avião :D
df_completo = pd.merge(
    df_missoes, 
    df_avioes[['aircraft', 'aircraft_type']], 
    how='left', 
    left_on='mds', 
    right_on='aircraft'
)

# Cruzando com as armas 
df_completo = pd.merge(
    df_completo,
    df_armas[['weapon_name', 'weapon_class']],
    how='left',
    left_on='type_of_frag',
    right_on='weapon_name'
)

In [148]:
# Tirando valor nulo de coordenadas
df_completo = df_completo.dropna(subset=['latitude', 'longitude'])

# Converter a string de data para o formato Datetime do Python
df_completo['msndate'] = pd.to_datetime(df_completo['msndate'], errors='coerce')

# Novas variáveis
df_completo['Ano'] = df_completo['msndate'].dt.year
df_completo['Mes'] = df_completo['msndate'].dt.month

# Classificar o peso do ataque
df_completo['Intensidade_Ataque'] = pd.cut(
    df_completo['total_tons'], 
    bins=[0, 5, 20, float('inf')], 
    labels=['Leve (< 5t)', 'Médio (5-20t)', 'Pesado (> 20t)']
)

In [149]:
# Colocando coisas nos nulos
df_completo['theater'] = df_completo['theater'].fillna('Não Informado')
df_completo['aircraft_type'] = df_completo['aircraft_type'].fillna('Outros')
df_completo['weapon_class'] = df_completo['weapon_class'].fillna('Não Identificado')


# INICIALIZAÇÃO DO APP E CONFIGURAÇÃO DO LAYOUT


In [150]:
app = dash.Dash(__name__, title="Dashboard THOR WWII", suppress_callback_exceptions=True)

app.layout = html.Div(id='main-container', children=[
    html.Header(style={'borderBottom': '1px solid #777', 'paddingBottom': '15px', 'marginBottom': '20px', 'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center'}, children=[
        html.Div([
            html.H1("Histórico de Operações Aéreas — Segunda Guerra Mundial", style={'margin': '0 0 5px 0'}),
            html.P("Análise descritiva orientada a insights baseada no framework THOR", style={'margin': '0', 'opacity': '0.7'})
        ]),
        # O Botão para mudar de cor
        dcc.RadioItems(
            id='tema-toggle',
            options=[
                {'label': ' ☀️ ', 'value': 'light'},
                {'label': ' 🌙 ', 'value': 'dark'}
            ],
            value='light',
            inline=True,
            style={'fontWeight': 'bold', 'fontSize': '18px', 'cursor': 'pointer'}
        )
    ]),
    
    dcc.Tabs(id="abas-navegacao", value='aba-executiva', children=[
        dcc.Tab(label='Dashboard 1 — Visão Geral Executiva', value='aba-executiva'),
        dcc.Tab(label='Dashboard 2 — Exploração Interativa', value='aba-exploratoria'),
    ]),
    
    html.Div(id='painel-conteudo')
])


# CRIANDO OS DASHBOARDS


In [151]:
@app.callback(
    Output('main-container', 'style'),
    Input('tema-toggle', 'value')
)
def atualizar_fundo_geral(tema):
    if tema == 'dark':
        return {'backgroundColor': '#111111', 'color': '#ffffff', 'minHeight': '100vh', 'fontFamily': 'sans-serif', 'padding': '20px', 'transition': '0.3s'}
    return {'backgroundColor': '#ffffff', 'color': '#000000', 'minHeight': '100vh', 'fontFamily': 'sans-serif', 'padding': '20px', 'transition': '0.3s'}


# Renderiza as Abas e aplica as cores nos Cards
@app.callback(
    Output('painel-conteudo', 'children'),
    [Input('abas-navegacao', 'value'),
     Input('tema-toggle', 'value')] # O tema entra aqui para pintar os Cards
)
def renderizar_aba(aba_selecionada, tema):
    # Lógica de cores baseada no tema
    bg_card = '#222222' if tema == 'dark' else '#ffffff'
    borda_card = '#444444' if tema == 'dark' else '#eeeeee'
    template_grafico = 'plotly_dark' if tema == 'dark' else 'simple_white'
    
    if aba_selecionada == 'aba-executiva':
        # DASHBOARD 1: 
        # Cálculos de KPI fixos para o painel executivo 
        total_missoes = f"{len(df_completo):,}".replace(",", ".")
        total_toneladas = f"{int(df_completo['total_tons'].sum()):,}".replace(",", ".")
        paises_atingidos = df_completo['tgt_country'].nunique()

        # Evolução temporal anual
        df_linha_ano = df_completo.groupby('Ano')['total_tons'].sum().reset_index()
        fig_temporal_sintetica = px.line(
            df_linha_ano, x='Ano', y='total_tons',
            title='Curva de Intensidade Logística: Toneladas Lançadas por Ano (1939-1945)',
            labels={'total_tons': 'Toneladas de Bombas', 'Ano': 'Ano do Conflito'}
        )
        fig_temporal_sintetica.update_layout(template=template_grafico)
        
        # Indicadores dos cards
        return html.Div(style={'paddingTop': '20px'}, children=[
            html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '30px'}, children=[
                html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '25px', 'borderRadius': '8px'}, children=[
                    html.H4("VOLUME DE MISSÕES", style={'margin': '0 0 10px 0', 'fontSize': '12px', 'opacity': '0.7', 'letterSpacing': '1px'}),
                    html.P(total_missoes, style={'margin': '0', 'fontSize': '32px', 'fontWeight': 'bold'})
                ]),
                html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '25px', 'borderRadius': '8px'}, children=[
                    html.H4("CARGA DESTRUTIVA", style={'margin': '0 0 10px 0', 'fontSize': '12px', 'opacity': '0.7', 'letterSpacing': '1px'}),
                    html.P(f"{total_toneladas} Tons", style={'margin': '0', 'fontSize': '32px', 'fontWeight': 'bold'})
                ]),
                html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '25px', 'borderRadius': '8px'}, children=[
                    html.H4("ALVOS GEOGRÁFICOS (PAÍSES)", style={'margin': '0 0 10px 0', 'fontSize': '12px', 'opacity': '0.7', 'letterSpacing': '1px'}),
                    html.P(paises_atingidos, style={'margin': '0', 'fontSize': '32px', 'fontWeight': 'bold'})
                ]),
            ]),
            
            html.Div(style={'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '20px', 'borderRadius': '8px'}, children=[
                dcc.Graph(figure=fig_temporal_sintetica)
            ])
        ])
        
    elif aba_selecionada == 'aba-exploratoria':
        # DASHBOARD 2:
        lista_teatros = sorted(df_completo['theater'].unique())
        lista_intensidades = sorted(df_completo['Intensidade_Ataque'].dropna().unique())
        
        # Cor de fundo pros dropdowns não ficarem com texto branco no fundo branco
        estilo_filtro = {'color': '#000000'} # O texto dentro do dropdown precisa ser escuro sempre
        
        return html.Div(style={'paddingTop': '20px'}, children=[
            # Painel lateral dos filtros(Teatro de Operações e Intensidade do Ataque)
            html.Div(style={'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '20px', 'borderRadius': '8px', 'marginBottom': '20px'}, children=[
                html.H3("Filtros de Segmentação Tática", style={'margin': '0 0 15px 0', 'fontSize': '16px'}),
                
                html.Div(style={'display': 'flex', 'gap': '30px'}, children=[
                    html.Div(style={'flex': '1'}, children=[
                        html.Label("Teatro de Operações:", style={'fontWeight': 'bold', 'display': 'block', 'marginBottom': '8px'}),
                        dcc.Dropdown(
                            id='filtro-theater',
                            options=[{'label': t, 'value': t} for t in lista_teatros],
                            value=lista_teatros[0],
                            clearable=False,
                            style=estilo_filtro
                        )
                    ]),
                    html.Div(style={'flex': '1'}, children=[
                        html.Label("Intensidade do Ataque (Carga de Bombas):", style={'fontWeight': 'bold', 'display': 'block', 'marginBottom': '8px'}),
                        dcc.Dropdown(
                            id='filtro-intensidade',
                            options=[{'label': i, 'value': i} for i in lista_intensidades],
                            value=[lista_intensidades[1], lista_intensidades[2]], 
                            multi=True,
                            style=estilo_filtro
                        )
                    ])
                ])
            ]),
            
            html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr', 'gap': '20px'}, children=[
                # O Mapa
                html.Div(style={'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                    dcc.Graph(id='grafico-mapa-alvos')
                ]),

                # Ranking dos paises e Tipos de Aviões
                html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
                    html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                        dcc.Graph(id='grafico-ranking-paises')
                    ]),
                    html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                        dcc.Graph(id='grafico-tipos-aeronaves')
                    ])
                ]),
                # Sazonalidade por mês e Altidade vs Carga
                html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
                    html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                        dcc.Graph(id='grafico-sazonalidade-mes')
                    ]),
                    html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                        dcc.Graph(id='grafico-dispersao-altitude')
                    ])
                ])
            ])
        ])

# CALLBACK DO DASHBOARD

In [152]:
@app.callback(
    [Output('grafico-mapa-alvos', 'figure'),
     Output('grafico-ranking-paises', 'figure'),
     Output('grafico-tipos-aeronaves', 'figure'),
     Output('grafico-sazonalidade-mes', 'figure'),
     Output('grafico-dispersao-altitude', 'figure')],
    [Input('filtro-theater', 'value'),
     Input('filtro-intensidade', 'value'),
     Input('tema-toggle', 'value')] 
)
def atualizar_dashboard_exploratorio(teatro_sel, intensidade_sel, tema):
    template_grafico = 'plotly_dark' if tema == 'dark' else 'simple_white'
    
    df_filtrado = df_completo[df_completo['theater'] == teatro_sel]
    if intensidade_sel:
        df_filtrado = df_filtrado[df_filtrado['Intensidade_Ataque'].isin(intensidade_sel)]
    
    # Amostra inteligente pra não quebrar a renderização
    df_mapa = df_filtrado.sample(n=min(5000, len(df_filtrado)), random_state=42) if len(df_filtrado) > 0 else df_filtrado

    # Gráfico de Dispersão Espacial (Mapa de Alvos)
    fig_mapa = px.scatter_geo(
        df_mapa, lat='latitude', lon='longitude', size='total_tons', color='Intensidade_Ataque',
        title=f'Mapeamento Geográfico — Teatro {teatro_sel}', projection="natural earth"
    )
    fig_mapa.update_layout(template=template_grafico, margin=dict(l=0, r=0, t=40, b=0))

    # Gráfico de Barras Horizontais (Ranking de Países Alvo)
    df_paises = df_filtrado.groupby('tgt_country')['total_tons'].sum().reset_index().sort_values(by='total_tons', ascending=True).tail(10)
    fig_barras = px.bar(df_paises, x='total_tons', y='tgt_country', orientation='h', title='Top 10 Países Alvo')
    fig_barras.update_layout(template=template_grafico)

    # Gráfico de Rosca (Tipos de Aeronaves Executoras)
    df_tipo_aviao = df_filtrado.groupby('aircraft_type').size().reset_index(name='Contagem')
    fig_rosca = px.pie(df_tipo_aviao, values='Contagem', names='aircraft_type', hole=0.4, title='Distribuição de Surtidas')
    fig_rosca.update_layout(template=template_grafico)

    # Gráfico de Barras (Sazonalidade Mensal Acumulada)
    df_sazonal = df_filtrado.groupby('Mes')['total_tons'].sum().reset_index()
    fig_sazonalidade = px.bar(df_sazonal, x='Mes', y='total_tons', title='Sazonalidade Mensal')
    fig_sazonalidade.update_xaxes(dtick=1)
    fig_sazonalidade.update_layout(template=template_grafico)

    # Gráfico de Dispersão Comparativo (Altitude Operacional vs Peso das Bombas)
    # Filtrando altitudes nulas ou inconsistentes
    df_altitude = df_filtrado[(df_filtrado['altitude_feet'] > 0) & (df_filtrado['total_tons'] > 0)]
    df_sub_altitude = df_altitude.sample(n=min(1000, len(df_altitude)), random_state=42) if len(df_altitude) > 0 else df_altitude
    fig_dispersao = px.scatter(df_sub_altitude, x='altitude_feet', y='total_tons', color='aircraft_type', title='Altitude vs Carga')
    fig_dispersao.update_layout(template=template_grafico)

    return fig_mapa, fig_barras, fig_rosca, fig_sazonalidade, fig_dispersao

In [153]:
if __name__ == '__main__':
    app.run(jupyter_mode="external", port=8052, debug=False)# ngrok http 8052(Se for usar ngrok por algum motivo)


Dash app running on http://127.0.0.1:8052/
